# Activation functions (`nn."function"`)

The most famous activation functions can be found in the `nn.`. And they can be divided by families, let's see some:

### 1. Probability functions

Probabilistic activation functions are functions that introduce randomness into the behavior of the neuron. Instead of always producing the same output for a given input, the output relies on a probability distribution. So, in one execution the neuron can activate, and in another it cannot.

1. `nn.Sigmoid()`: Recives any value and returns a number between 0 and 1 
2. `nn.Softmax()`: Unlike the sigmoid, softmax needs a vector of numbers and will transform all outputs into probabilities whose sum needs to be 1
3. `nn.Tanh()`: Recives any value and returns a number between -1 and 1 (looks like sigmoid) 

### 2. Deterministics Functions (ReLU family)

For the same input, the output is always the same. The most common difference between all it's about negative numbers. Here we need to understand a concept, in some architectures, negative values can be treated as unhelpful contributions or as contrary evidence or even 'noisiness', being discarded to simplify network learning.

1. `nn.ReLU()`: Negatives become 0, positives become them. Does not leave negatives $f(x)=max(0,x)$. Discard all the 'noise'
2. `nn.LeakyReLU`: Allows small negatives numbers, a minuscule noise 
$f(x)= x, x>0$ and
$f(x)= ax, x<0$ wheres the $a$ it's the alpha and we can define (normally 0.01)
3. `nn.GELU()`: Allows negative numbers. The closer to the positive side and the higher the positive value, the more it passes
4. `nn.SiLU()`: Allows negative numbers, have a hyperparameter alpha too but in this case it's a sigmoid function $SiLU(x)=x⋅σ(x)$. The number will pass by a sigmoid where will calculate their importance and then will dot by the number original.
5. `nn.ELU()`: Was created as an alternative to ReLU to preserve negative values without using a constant slope like Leaky ReLU.
6. `nn.SELU()`: A evolution of ELU. Allow negative values, but in a controlled manner, while keeping network activations naturally stable. , statistically, the function itself tends to push activations to average ≈ 0 and variance ≈ 1

Here have a table with all the most famous deterministics function, and how much their pass of the original value

| Values Negative (More → Less) | Values Positive (More → Less) |
|-------------------------------|-------------------------------|
| 1. **SELU / ELU**             | 1. **SELU**                   |
| 2. **SiLU**                   | 2. **ReLU / Leaky ReLU / ELU**|
| 3. **GELU**                   | 3. **GELU**                   |
| 4. **Leaky ReLU**             | 4. **SiLU**                   |
| 5. **ReLU**                   | 5. **Tanh / Sigmoid**         |


# Summary

| Activation | Treatment of Negative Values | Main Advantage | Main Drawback |
|------------|-----------------------------|----------------|---------------|
| **ReLU** | Removes all negative activations, treating them as noise or unhelpful contributions | Produces sparse representations, reduces noise, and is extremely fast | Can discard useful contrary evidence and suffer from dead neurons |
| **Leaky ReLU** | Preserves a small fraction of negative information | Reduces information loss and prevents dead neurons while keeping ReLU's simplicity | Requires manually choosing α and still suppresses most negative evidence |
| **ELU** | Preserves negative evidence with saturation | Richer information flow, zero-centered activations, and improved optimization stability | More computationally expensive due to the exponential term |
| **SELU** | Preserves negative information while keeping activations statistically balanced | Self-normalization, stable deep networks, and reduced need for normalization layers | Requires specific initialization and architectural assumptions |
| **GELU** | Smoothly attenuates negative values instead of removing them | Filters noise softly while preserving potentially useful information; excellent gradient flow | More complex and computationally heavier than ReLU |
| **SiLU (Swish)** | Adaptively preserves negative values according to their importance | Rich information flow, adaptive gating, and minimal information loss | Less sparse representations and higher computational cost |

---

# Intuition

```text
ReLU → "Negative values are probably noise. (CNNs and Classic Networks)"
Leaky ReLU → "Negative values are mostly noise, but keep a tiny amount just in case"
ELU → "Negative evidence is useful, but should be bounded. (Deep MLPs)"
SELU → "Negative evidence is useful and should help keep the network stable"
GELU → "Negative values may contain information; attenuate them smoothly. (Transformers)"
SiLU → "Every activation deserves a contribution proportional to its importance. (Modern Vision Models)"
```

In [21]:
import torch
import torch.nn.functional as F
import torch.nn as nn
torch.set_printoptions(precision=5, sci_mode=False)

tensor = torch.tensor([-4, 100, 10, -3, 1], dtype=torch.float32)

# Probabilistic-style activations
sigmoid = F.sigmoid(tensor)
softmax = F.softmax(tensor, dim=0)
tanh = F.tanh(tensor)

# Deterministic activations
relu = F.relu(tensor)
leaky_relu = F.leaky_relu(tensor, negative_slope=0.01)
gelu = F.gelu(tensor)
silu = F.silu(tensor)
elu = F.elu(tensor, alpha=1.0)
selu = F.selu(tensor)

print("=" * 50)
print(f"Input       : {tensor}")

print("\n--- Probabilistic Activations ---")
print(f"Sigmoid     : {sigmoid}")
print(f"Softmax     : {softmax}")
print(f"Tanh        : {tanh}")

print("\n--- Deterministic Activations ---")
print(f"ReLU        : {relu}")
print(f"Leaky ReLU  : {leaky_relu}")
print(f"GELU        : {gelu}")
print(f"SiLU        : {silu}")
print(f"ELU         : {elu}")
print(f"SELU        : {selu}")

print("=" * 50)



"""
OR WE CAN USE IN THE NN.MODULE FORM

"""


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(5, 5)
        self.gelu = nn.GELU()

    def forward(self, x):
        x = self.fc(x)
        x = self.gelu(x)
        return x
    
model = Model()
output = model(tensor)
print(F'nnModule: {output}')

Input       : tensor([ -4., 100.,  10.,  -3.,   1.])

--- Probabilistic Activations ---
Sigmoid     : tensor([0.01799, 1.00000, 0.99995, 0.04743, 0.73106])
Softmax     : tensor([0.00000, 1.00000, 0.00000, 0.00000, 0.00000])
Tanh        : tensor([-0.99933,  1.00000,  1.00000, -0.99505,  0.76159])

--- Deterministic Activations ---
ReLU        : tensor([  0., 100.,  10.,   0.,   1.])
Leaky ReLU  : tensor([ -0.04000, 100.00000,  10.00000,  -0.03000,   1.00000])
GELU        : tensor([ -0.00013, 100.00000,  10.00000,  -0.00405,   0.84134])
SiLU        : tensor([ -0.07194, 100.00000,   9.99955,  -0.14228,   0.73106])
ELU         : tensor([ -0.98168, 100.00000,  10.00000,  -0.95021,   1.00000])
SELU        : tensor([ -1.72590, 105.07010,  10.50701,  -1.67057,   1.05070])
nnModule: tensor([ 3.35911,  0.00000, 25.91530,  5.96314,  0.00000],
       grad_fn=<GeluBackward0>)
